## SRP541763

**paper:** [PMID: 39587248](https://pmc.ncbi.nlm.nih.gov/articles/PMC11589579/) - cis- and trans-regulatory contributions to a hierarchy of factors influencing gene expression variation, 2024

**date, curator:** 2026-06-30, Sara Carsanaro

**notes**
* removed samples on Octanoic Acid Media current or previous generation - not healthy wild type
* all samples are female
* adults are 0-4 days post eclosion, larvae are L3
* white501 / Tucson 14021-0251.195 and 14021-0428.25 (which is maybe not the same as Rob3c / Tucson 14021-0248.25, very confusing)
* sequencing and library prep not clear - University of Michigan Medical School DNA Sequencing Core - standard Illumina protocols

### annotation summary

In [24]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Larvae,UBERON:0002548,larva,perfect match
9,Adult,UBERON:0007023,adult organism,perfect match


In [25]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Larvae L3,UBERON:8000002,third instar larva stage,perfect match
9,Adult,UBERON:0000066,fully formed stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP541763"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX26528554,SRP541763,NextSeq 550,SRS23038603,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599962,Larvae,Larvae L3,,,,,,,1489344,,,,,,Larvae_Hybrid_SS3,SAMN44489351,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
1,SRX26528553,SRP541763,NextSeq 550,SRS23038602,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599961,Larvae,Larvae L3,,,,,,,1489344,,,,,,Larvae_Hybrid_SS2,SAMN44489352,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
2,SRX26528552,SRP541763,NextSeq 550,SRS23038601,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599960,Larvae,Larvae L3,,,,,,,1489344,,,,,,Larvae_Hybrid_SS1,SAMN44489353,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
3,SRX26528542,SRP541763,NextSeq 550,SRS23038591,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599950,Larvae,Larvae L3,,,,,14021-0428.25,,7238,,,,,,Larvae_Sec_SS3,SAMN44489363,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
4,SRX26528541,SRP541763,NextSeq 550,SRS23038590,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599949,Larvae,Larvae L3,,,,,14021-0428.25,,7238,,,,,,Larvae_Sec_SS2,SAMN44489364,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
5,SRX26528540,SRP541763,NextSeq 550,SRS23038589,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599948,Larvae,Larvae L3,,,,,14021-0428.25,,7238,,,,,,Larvae_Sec_SS1,SAMN44489365,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
6,SRX26528530,SRP541763,NextSeq 550,SRS23038578,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599938,Larvae,Larvae L3,,,,,white501 / Tucson 14021-0251.195,,7240,,,,,,Larvae_Sim_SS3,SAMN44489375,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
7,SRX26528529,SRP541763,NextSeq 550,SRS23038577,,

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Adult' 'Larvae']


In [6]:

# Larvae
library.loc[library["infoOrgan"] == "Larvae", "anatId"] = "UBERON:0002548"
library.loc[library["infoOrgan"] == "Larvae", "anatName"] = "larva"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Larvae", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Larvae", "anatBiologicalStatus"] = "full sampling"

# Adult
library.loc[library["infoOrgan"] == "Adult", "anatId"] = "UBERON:0007023"
library.loc[library["infoOrgan"] == "Adult", "anatName"] = "adult organism"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Adult", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Adult", "anatBiologicalStatus"] = "full sampling"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX26528554,SRP541763,NextSeq 550,SRS23038603,UBERON:0002548,larva,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599962,Larvae,Larvae L3,perfect match,full sampling,,,,,1489344,,,,,,Larvae_Hybrid_SS3,SAMN44489351,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
1,SRX26528553,SRP541763,NextSeq 550,SRS23038602,UBERON:0002548,larva,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599961,Larvae,Larvae L3,perfect match,full sampling,,,,,1489344,,,,,,Larvae_Hybrid_SS2,SAMN44489352,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
2,SRX26528552,SRP541763,NextSeq 550,SRS23038601,UBERON:0002548,larva,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599960,Larvae,Larvae L3,perfect match,full sampling,,,,,1489344,,,,,,Larvae_Hybrid_SS1,SAMN44489353,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
3,SRX26528542,SRP541763,NextSeq 550,SRS23038591,UBERON:0002548,larva,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599950,Larvae,Larvae L3,perfect match,full sampling,,,14021-0428.25,,7238,,,,,,Larvae_Sec_SS3,SAMN44489363,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
4,SRX26528541,SRP541763,NextSeq 550,SRS23038590,UBERON:0002548,larva,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599949,Larvae,Larvae L3,perfect match,full sampling,,,14021-0428.25,,7238,,,,,,Larvae_Sec_SS2,SAMN44489364,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
5,SRX26528540,SRP541763,NextSeq 550,SRS23038589,UBERON:0002548,larva,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599948,Larvae,Larvae L3,perfect match,full sampling,,,14021-0428.25,,7238,,,,,,Larvae_Sec_SS1,SAMN44489365,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
6,SRX26528530,SRP541763,NextSeq 550,SRS23038578,UBERON:0002548,larva,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599938,Larvae,Larvae L3,perfect match,full sampling,,,white501 / Tucson 14021-0251.195,,7240,,,,,,Larvae_Sim_SS3,SAMN44489375,,,,,,,,,,,30/06/2026,RNA was ext

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['Adult' 'Larvae L3']


In [8]:
# Larvae
library.loc[library["infoStage"] == "Larvae L3", "stageId"] = "UBERON:8000002"
library.loc[library["infoStage"] == "Larvae L3", "stageName"] = "third instar larva stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "Larvae L3", "stageAnnotationStatus"] = "perfect match"

# conditional (based off infoStage)
library.loc[library["infoStage"] == "Adult", "stageId"] = "UBERON:0000066"
library.loc[library["infoStage"] == "Adult", "stageName"] = "fully formed stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "Adult", "stageAnnotationStatus"] = "perfect match"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX26528554,SRP541763,NextSeq 550,SRS23038603,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599962,Larvae,Larvae L3,perfect match,full sampling,perfect match,,,,1489344,,,,,,Larvae_Hybrid_SS3,SAMN44489351,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
1,SRX26528553,SRP541763,NextSeq 550,SRS23038602,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599961,Larvae,Larvae L3,perfect match,full sampling,perfect match,,,,1489344,,,,,,Larvae_Hybrid_SS2,SAMN44489352,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
2,SRX26528552,SRP541763,NextSeq 550,SRS23038601,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599960,Larvae,Larvae L3,perfect match,full sampling,perfect match,,,,1489344,,,,,,Larvae_Hybrid_SS1,SAMN44489353,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
3,SRX26528542,SRP541763,NextSeq 550,SRS23038591,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599950,Larvae,Larvae L3,perfect match,full sampling,perfect match,,14021-0428.25,,7238,,,,,,Larvae_Sec_SS3,SAMN44489363,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
4,SRX26528541,SRP541763,NextSeq 550,SRS23038590,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599949,Larvae,Larvae L3,perfect match,full sampling,perfect match,,14021-0428.25,,7238,,,,,,Larvae_Sec_SS2,SAMN44489364,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
5,SRX26528540,SRP541763,NextSeq 550,SRS23038589,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599948,Larvae,Larvae L3,perfect match,full sampling,perfect match,,14021-0428.25,,7238,,,,,,Larvae_Sec_SS1,SAMN44489365,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [9]:
library.loc[:,'sex'] = 'F'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX26528554,SRP541763,NextSeq 550,SRS23038603,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599962,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,,,,Larvae_Hybrid_SS3,SAMN44489351,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
1,SRX26528553,SRP541763,NextSeq 550,SRS23038602,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599961,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,,,,Larvae_Hybrid_SS2,SAMN44489352,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
2,SRX26528552,SRP541763,NextSeq 550,SRS23038601,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599960,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,,,,Larvae_Hybrid_SS1,SAMN44489353,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
3,SRX26528542,SRP541763,NextSeq 550,SRS23038591,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599950,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,,,,Larvae_Sec_SS3,SAMN44489363,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
4,SRX26528541,SRP541763,NextSeq 550,SRS23038590,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599949,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,,,,Larvae_Sec_SS2,SAMN44489364,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
5,SRX26528540,SRP541763,NextSeq 550,SRS23038589,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599948,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,,,,Larvae_Sec_SS1,SAMN44489365,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX26528554,SRP541763,NextSeq 550,SRS23038603,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599962,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS3,SAMN44489351,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
1,SRX26528553,SRP541763,NextSeq 550,SRS23038602,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599961,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS2,SAMN44489352,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
2,SRX26528552,SRP541763,NextSeq 550,SRS23038601,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599960,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS1,SAMN44489353,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
3,SRX26528542,SRP541763,NextSeq 550,SRS23038591,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599950,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS3,SAMN44489363,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
4,SRX26528541,SRP541763,NextSeq 550,SRS23038590,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599949,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS2,SAMN44489364,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
5,SRX26528540,SRP541763,NextSeq 550,SRS23038589,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599948,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS1,SAMN44489365,,,,,,,,,,,30/06/2026,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard 

#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-07-01'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX26528554,SRP541763,NextSeq 550,SRS23038603,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599962,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS3,SAMN44489351,,,,,,,,,,SAC,2026-07-01,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
1,SRX26528553,SRP541763,NextSeq 550,SRS23038602,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599961,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS2,SAMN44489352,,,,,,,,,,SAC,2026-07-01,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
2,SRX26528552,SRP541763,NextSeq 550,SRS23038601,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599960,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS1,SAMN44489353,,,,,,,,,,SAC,2026-07-01,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
3,SRX26528542,SRP541763,NextSeq 550,SRS23038591,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599950,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS3,SAMN44489363,,,,,,,,,,SAC,2026-07-01,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
4,SRX26528541,SRP541763,NextSeq 550,SRS23038590,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599949,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS2,SAMN44489364,,,,,,,,,,SAC,2026-07-01,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequencing using standard Illumina protocols,,,,Larvae,,Larvae,,TRANSCRIPTOMIC,cDNA
5,SRX26528540,SRP541763,NextSeq 550,SRS23038589,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599948,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS1,SAMN44489365,,,,,,,,,,SAC,2026-07-01,RNA was extracted from a single tissue homogenate of either 10 whole adult females or 10 whole female L3 larvae using the Promega SV total RNA extraction system RNA libraries were prepared for sequenci

#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 39587248'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX26528554,SRP541763,NextSeq 550,SRS23038603,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599962,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS3,SAMN44489351,,,,,,,PMID: 39587248,,,SAC,2026-07-01
1,SRX26528553,SRP541763,NextSeq 550,SRS23038602,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599961,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS2,SAMN44489352,,,,,,,PMID: 39587248,,,SAC,2026-07-01
2,SRX26528552,SRP541763,NextSeq 550,SRS23038601,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599960,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS1,SAMN44489353,,,,,,,PMID: 39587248,,,SAC,2026-07-01
3,SRX26528542,SRP541763,NextSeq 550,SRS23038591,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599950,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS3,SAMN44489363,,,,,,,PMID: 39587248,,,SAC,2026-07-01
4,SRX26528541,SRP541763,NextSeq 550,SRS23038590,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599949,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS2,SAMN44489364,,,,,,,PMID: 39587248,,,SAC,2026-07-01
5,SRX26528540,SRP541763,NextSeq 550,SRS23038589,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599948,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS1,SAMN44489365,,,,,,,PMID: 39587248,,,SAC,2026-07-01
6,SRX26528530,SRP541763,NextSeq 550,SRS23038578,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599938,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,white501 / Tucson 14021-0251.195,,7240,,,polyA,,,Larvae_Sim_SS3,SAMN44489375,,,,,,,PMID: 39587248,,,SAC,2026-07-01
7,SRX26528529,SRP541763,NextSeq 550,SRS23038577,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599937,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,white501 / Tucson 14021-0251.195,,7240,,,polyA,,,Larvae_Sim_SS2,SAMN44489376,,,,,,,PMID: 39587248,,,SAC,2026-07-01
8,SRX26528528,SRP541763,NextSeq 550,SRS23038576,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599936,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,white501 / Tucson 14021-0251.195,,7240,,,polyA,,,Larvae_Sim_SS1,SAMN44489377,,,,,,,PMID: 39587248,,,SAC,2026-07-01
9,SRX26528518,SRP541763,NextSeq 550,SRS23038566,UBERON:0007023,adult organism,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8599926,Adult,Adult,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Adult_Hybrid_SS3,SAMN44489387,0-4,day post eclosion,,,,,PMID: 39587248,,,SAC,2026-07-01


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP541763,cis- and trans-regulatory contributions to a hierarchy of factors influencing gene expression variation,"Gene expression variation results from numerous sources including genetic, environmental, life stage, and even the environment experienced by previous generations. While the importance of each has been demonstrated in diverse organisms, their relative contributions remain understudied because few investigations have simultaneously determined each within a single experiment. Here we quantified genome-wide gene expression traits in Drosophila, quantified the contribution of multiple different sources of trait variation and determined the molecular mechanisms underlying observed variation. Our results show that there is a clear hierarchy in our data with genome and developmental stage contributing on average considerably more than current and finally previous generation environmental effects. We also determined the role of cis and trans-regulatory changes across different sources of trait variation, highlighting their importance in adaptation and environmental responses and showing unexpectedly that transgenerational effects herein were predominantly associated with changes in trans-regulation. Overall design: Differential gene expression analysis of D. sechellia and D. simulans, and D. sechellia x D. simulan hybrid larval and adult flies exposed to standard media and food supplemented with octanoic acid in both the current generation and previous generation using RNA-sequencing.",SRA,,,,,,,PRJNA1179363,39587248,,10.1038/s42003-024-07255-6,,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

18

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP541763,cis- and trans-regulatory contributions to a hierarchy of factors influencing gene expression variation,"Gene expression variation results from numerous sources including genetic, environmental, life stage, and even the environment experienced by previous generations. While the importance of each has been demonstrated in diverse organisms, their relative contributions remain understudied because few investigations have simultaneously determined each within a single experiment. Here we quantified genome-wide gene expression traits in Drosophila, quantified the contribution of multiple different sources of trait variation and determined the molecular mechanisms underlying observed variation. Our results show that there is a clear hierarchy in our data with genome and developmental stage contributing on average considerably more than current and finally previous generation environmental effects. We also determined the role of cis and trans-regulatory changes across different sources of trait variation, highlighting their importance in adaptation and environmental responses and showing unexpectedly that transgenerational effects herein were predominantly associated with changes in trans-regulation. Overall design: Differential gene expression analysis of D. sechellia and D. simulans, and D. sechellia x D. simulan hybrid larval and adult flies exposed to standard media and food supplemented with octanoic acid in both the current generation and previous generation using RNA-sequencing.",SRA,partial,Bgee 1K,18,,,,PRJNA1179363,39587248,,10.1038/s42003-024-07255-6,,


#### paper and xrefs

In [19]:
experiment.loc[:,'GSE'] = 'GSE280534'
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '39587248'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11589579/'
experiment.loc[:,'DOI'] = '10.1038/s42003-024-07255-6'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP541763,cis- and trans-regulatory contributions to a hierarchy of factors influencing gene expression variation,"Gene expression variation results from numerous sources including genetic, environmental, life stage, and even the environment experienced by previous generations. While the importance of each has been demonstrated in diverse organisms, their relative contributions remain understudied because few investigations have simultaneously determined each within a single experiment. Here we quantified genome-wide gene expression traits in Drosophila, quantified the contribution of multiple different sources of trait variation and determined the molecular mechanisms underlying observed variation. Our results show that there is a clear hierarchy in our data with genome and developmental stage contributing on average considerably more than current and finally previous generation environmental effects. We also determined the role of cis and trans-regulatory changes across different sources of trait variation, highlighting their importance in adaptation and environmental responses and showing unexpectedly that transgenerational effects herein were predominantly associated with changes in trans-regulation. Overall design: Differential gene expression analysis of D. sechellia and D. simulans, and D. sechellia x D. simulan hybrid larval and adult flies exposed to standard media and food supplemented with octanoic acid in both the current generation and previous generation using RNA-sequencing.",SRA,partial,Bgee 1K,18,,,GSE280534,PRJNA1179363,39587248,https://pmc.ncbi.nlm.nih.gov/articles/PMC11589579/,10.1038/s42003-024-07255-6,,


#### comments

In [20]:
experiment.loc[:,'comment'] = 'removed libraries on octanoic acid media'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP541763,cis- and trans-regulatory contributions to a hierarchy of factors influencing gene expression variation,"Gene expression variation results from numerous sources including genetic, environmental, life stage, and even the environment experienced by previous generations. While the importance of each has been demonstrated in diverse organisms, their relative contributions remain understudied because few investigations have simultaneously determined each within a single experiment. Here we quantified genome-wide gene expression traits in Drosophila, quantified the contribution of multiple different sources of trait variation and determined the molecular mechanisms underlying observed variation. Our results show that there is a clear hierarchy in our data with genome and developmental stage contributing on average considerably more than current and finally previous generation environmental effects. We also determined the role of cis and trans-regulatory changes across different sources of trait variation, highlighting their importance in adaptation and environmental responses and showing unexpectedly that transgenerational effects herein were predominantly associated with changes in trans-regulation. Overall design: Differential gene expression analysis of D. sechellia and D. simulans, and D. sechellia x D. simulan hybrid larval and adult flies exposed to standard media and food supplemented with octanoic acid in both the current generation and previous generation using RNA-sequencing.",SRA,partial,Bgee 1K,18,,,GSE280534,PRJNA1179363,39587248,https://pmc.ncbi.nlm.nih.gov/articles/PMC11589579/,10.1038/s42003-024-07255-6,,removed libraries on octanoic acid media


#### save complete file

In [21]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [22]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [23]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 9
Errors: 9
Warnings: 0
Top codes:
  - BAD_VALUE: 9


#### check columns match

In [26]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [27]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
66856,SRX12281177,SRP338066,Illumina HiSeq 2000,SRS10255124,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,6-10 day old adult,perfect match,not documented,perfect match,M,Tucson 14030-0811.24,,7260,TruSeq Stranded mRNA,full_length,polyA,,,Invertebrate sample from Drosophila willistoni,SAMN21533502,6-10,day post eclosion,,,EFO:0002951,virgin,no paper but appears to be same group as PMID:...,,virgin,SAC,2026-06-29
66857,SRX12281176,SRP338066,Illumina HiSeq 2000,SRS10255123,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,6-10 day old adult,perfect match,not documented,perfect match,M,Tucson 14030-0811.24,,7260,TruSeq Stranded mRNA,full_length,polyA,,,Invertebrate sample from Drosophila willistoni,SAMN21533501,6-10,day post eclosion,,,EFO:0002951,virgin,no paper but appears to be same group as PMID:...,,virgin,SAC,2026-06-29
66858,SRX26528554,SRP541763,NextSeq 550,SRS23038603,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS3,SAMN44489351,,,,,,,PMID: 39587248,,,SAC,2026-07-01
66859,SRX26528553,SRP541763,NextSeq 550,SRS23038602,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS2,SAMN44489352,,,,,,,PMID: 39587248,,,SAC,2026-07-01
66860,SRX26528552,SRP541763,NextSeq 550,SRS23038601,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,,,1489344,,,polyA,,,Larvae_Hybrid_SS1,SAMN44489353,,,,,,,PMID: 39587248,,,SAC,2026-07-01
66861,SRX26528542,SRP541763,NextSeq 550,SRS23038591,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS3,SAMN44489363,,,,,,,PMID: 39587248,,,SAC,2026-07-01
66862,SRX26528541,SRP541763,NextSeq 550,SRS23038590,UBERON:0002548,larva,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Larvae,Larvae L3,perfect match,full sampling,perfect match,F,14021-0428.25,,7238,,,polyA,,,Larvae_Sec_SS2,SAMN44489364,,,,,,,PMID: 39587248,,,SAC,2026-07-01


In [28]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1275,SRP511279,Drosophila species HiC and RNA sequencing,We generated HiC data from whole female flies ...,SRA,partial,Bgee 1K,30,"Poly(A) mRNA Magnetic Isolation Module,NEBNext...",full_length,,PRJNA1119277,40102968,https://pmc.ncbi.nlm.nih.gov/articles/PMC11917...,10.1186/s13059-025-03527-4,,removed HiC libraries
1276,SRP338066,"Ovary, Testis and Accessory Gland sequencing o...",Purpose: Sequencing of gonads of Drosophila wi...,SRA,total,Bgee 1K,18,TruSeq Stranded mRNA,full_length,,PRJNA764902,,,,,cannot find paper but appears to be same group...
1277,SRP541763,cis- and trans-regulatory contributions to a h...,Gene expression variation results from numerou...,SRA,partial,Bgee 1K,18,,,GSE280534,PRJNA1179363,39587248,https://pmc.ncbi.nlm.nih.gov/articles/PMC11589...,10.1038/s42003-024-07255-6,,removed libraries on octanoic acid media


### add annotations to git

In [29]:
! git pull

Already up to date.


In [30]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [31]:
! git add $git_experiment_path $git_library_path

In [32]:
! git commit -m $commit_message_exp

[develop 390a764] adding annotated bulk experiment SRP541763
 2 files changed, 19 insertions(+)


In [33]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.20 KiB | 655.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   31b5ace..390a764  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push